In [ ]:
import pandas as pd
import numpy as np
import openpyxl
import re
from pathlib import Path
from datetime import date, timedelta
from natsort import natsort_keygen, natsorted, ns
from xlsxwriter.utility import xl_col_to_name
from datetime import datetime

import sys
sys.path.insert(0, str(Path("../src").resolve()))
import ado_reporting as cl
import daily_charge as dc
import work_items_report as wir
import data_validation as dv
import excel_import as xi
import burndown as bd

pd.options.display.max_columns = None  # type: ignore
pd.options.display.float_format = "{:,.2f}".format


In [2]:
# ==== USER CONFIG ====
project = "MarketRMS"

start_date = date(2026, 3, 1)  # <- report generated since that date

# today = date.today()

today = "2026-04-01"
today = datetime.strptime(today, "%Y-%m-%d").date()

# --- Validate 'today' ---
today = dv.validate_today(today)

DAYS_BACK = (today - start_date).days + 1

# Paths
BASE_DIR = Path("..") / "data"

# Categories
leaf_types = ["Task", "Bug"]
removed_state = "Removed"
doneish_statuses = {"Code Review", "Ready for UAT", "Ready for Prod", "In UAT", "Closed"}
daily_charge_output_path = BASE_DIR / f"{project}_daily_charge_report.xlsx"
today_ts = pd.Timestamp(today)
today_str = today.strftime("%Y-%m-%d")
output_file = BASE_DIR / f"{project}_ProjectStatus_{today_str}_ADO_API.xlsx"
burndown_file = BASE_DIR / f"{project}_Burndown_{today_str}.xlsx"


# Prepare shapshots dataset for analysis

In [3]:
wi_raw_snapshots = cl.get_wi_snapshots(
    project=project,
    days_back=DAYS_BACK,
    removed_state=removed_state,
)

# --- Validate source data freshness ---
_latest_snap = pd.Timestamp(wi_raw_snapshots["SnapshotDate"].max()).date()
if _latest_snap < today - timedelta(days=1):
    raise RuntimeError(
        "Project snapshots data is outdated. Please run synchronisation with ADO."
    )


Historical data selected: 2026-02-28 - 2026-03-31
Latest live snapshot file: MarketRMS_live_snapshot_2026-04-01.parquet (date=2026-04-01)
Live rows loaded: 1923
Final dataset date range: 2026-02-28 → 2026-04-01


# Identify related sprints

In [4]:
sprints, current_sprint, previous_sprint = cl.get_sprint_info(wi_raw_snapshots, today)

previous_sprint, current_sprint

('Sprint 35 (16.03-27.03)', 'Sprint 36 (30.03-10.04)')

# Charge by day report

In [5]:
# Compute daily charge tables (per assignee, per task) from snapshots.
# See src/daily_charge.py for the calculation.


In [6]:
# --- Daily charge by Assignee ---
daily_charge_by_person = dc.compute_daily_charge_by_person(wi_raw_snapshots)


In [7]:
# --- Daily charge by Task ---
daily_charge_by_task = dc.compute_daily_charge_by_task(wi_raw_snapshots)


In [8]:
# --- Download Daily charge for Power BI ---
daily_charge_by_person_bi = dc.export_daily_charge_for_powerbi(
    daily_charge_by_person,
    BASE_DIR / f"{project}_PowerBI_daily_charge.csv",
)


# Report for Daily (Stand-Ups)

In [9]:
# Build the daily meeting (stand-up) report.
# See src/work_items_report.py for the calculation.
daily_meeting_filtered, completed_cols = wir.compute_daily_meeting(
    wi_raw_snapshots, daily_charge_by_task, today
)
col_completed_earliest, col_completed_mid, col_completed_latest = completed_cols


# Sprint Burndown

In [ ]:
# Compute sprint burndowns (overall and per-assignee) from snapshots.
# See src/burndown.py for the calculation.
bd_current = bd.compute_sprint_burndown(wi_raw_snapshots, current_sprint)
bd_prev = bd.compute_sprint_burndown(wi_raw_snapshots, previous_sprint)


In [ ]:
# Per-assignee burndown for the current sprint (wide pivot for Excel).
bd_assignee_pivot = bd.compute_burndown_by_assignee(wi_raw_snapshots, current_sprint)


In [13]:
# Save the Team Analytics Dashboard workbook.
# See src/excel_import.py for the per-sheet writers and formatting rules.
xi.save_team_analytics_report(
    output_file,
    summary=daily_meeting_filtered,
    completed_cols=completed_cols,
    daily_charge_by_person=daily_charge_by_person,
    daily_charge_by_task=daily_charge_by_task,
    bd_current=bd_current,
    bd_prev=bd_prev,
    bd_assignee_pivot=bd_assignee_pivot,
    current_sprint=current_sprint,
    previous_sprint=previous_sprint,
)

print(f"\u2705 Done! File saved: {output_file}")


✅ Done! File saved: ../data/MarketRMS_ProjectStatus_2026-04-01_ADO_API.xlsx
